---
## Microlesson 1 — Editable install of `wdo`

Verify that `wdo` is imported from our source folder (not a frozen site-packages copy).
If `wdo.__file__` points inside `site-packages` instead of your source folder,
run `pip install -e Resources/wdo` from this directory.


In [1]:
import asyncio, sys
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

%load_ext autoreload
%autoreload 2

In [2]:
import wdo
print('wdo source:', wdo.__file__)

from wdo.geometry.distance import haversine_km
la_to_nyc = haversine_km((34.0, -118.0), (40.7, -74.0))
print(f'LA → NYC: {la_to_nyc:.1f} km  (expect ~3 940 km)')

wdo source: C:\Code\5993-Spatial-Data-and-Mapping\Assignments_Completed\04-Wordle\Resources\wdo\wdo\__init__.py
LA → NYC: 3918.2 km  (expect ~3 940 km)


---
## Microlesson 2 — Meet the data (ISO-2 / ISO-3 bridge)

Load the country polygons and the flag index, then join them by name.
The bridge lives in `wdo.io.country_lookup.ALIASES`.


In [3]:
import json
import pathlib

from wdo.io.geojson_tools import iter_features, load_geojson
from wdo.io.country_lookup import build_country_lookup

DATA_DIR = pathlib.Path('Resources/Data')
FLAG_DIR = DATA_DIR / 'flag-icons'

countries = load_geojson(DATA_DIR / 'countries_export.json')
print(f"Feature count : {countries['feature_count']}")
print(f"Property names: {countries['property_names']}")
print(f"First feature : {next(iter_features(countries))['properties']}")

Feature count : 255
Property names: ['ADMIN', 'ISO_A3']
First feature : {'ADMIN': 'Aruba', 'ISO_A3': 'ABW'}


In [4]:
with open(FLAG_DIR / 'country.json', encoding='utf-8') as f:
    flag_index = json.load(f)

print(f'Flag index entries: {len(flag_index)}')
print(f'Fields: {list(flag_index[0].keys())}')
print(f'Sample: {flag_index[0]}')

Flag index entries: 271
Fields: ['capital', 'code', 'continent', 'flag_1x1', 'flag_4x3', 'iso', 'name']
Sample: {'capital': 'Kabul', 'code': 'af', 'continent': 'Asia', 'flag_1x1': 'flags/1x1/af.svg', 'flag_4x3': 'flags/4x3/af.svg', 'iso': True, 'name': 'Afghanistan'}


In [5]:
country_lookup = build_country_lookup(countries, flag_index)

print(f'\nLookup size: {len(country_lookup)}')
print('France entry:', {k: v for k, v in country_lookup['FRA'].items() if k != 'feature'})
print('Brazil entry:', {k: v for k, v in country_lookup['BRA'].items() if k != 'feature'})

[build_country_lookup] 15 unmatched (no flag will show for these):
  Ashmore and Cartier Islands (-99)
  Bajo Nuevo Bank (Petrel Is.) (-99)
  Cyprus No Mans Area (-99)
  Coral Sea Islands (-99)
  Northern Cyprus (-99)
  Dhekelia Sovereign Base Area (-99)
  Indian Ocean Territories (-99)
  Baykonur Cosmodrome (-99)
  Siachen Glacier (-99)
  Spratly Islands (-99)
  Scarborough Reef (-99)
  Serranilla Bank (-99)
  Somaliland (-99)
  US Naval Base Guantanamo Bay (-99)
  Akrotiri Sovereign Base Area (-99)

Lookup size: 239
France entry: {'name': 'France', 'iso3': 'FRA', 'iso2': 'fr', 'flag_path': 'flags/4x3/fr.svg'}
Brazil entry: {'name': 'Brazil', 'iso3': 'BRA', 'iso2': 'br', 'flag_path': 'flags/4x3/br.svg'}


---
## Microlesson 3 — Draw a country on a map

Use `bbox_from_feature`, `make_map`, `add_geojson`, and `fit_map_to_geojson`
from `wdo`. Try Chile (thin + tall), Russia (huge + antimeridian), and Nauru (tiny).


In [6]:
from wdo.geometry.bbox import bbox_from_feature
from wdo.maps.leaflet_helpers import add_geojson, fit_map_to_geojson, make_map

feature_by_iso3 = {f['properties']['ISO_A3']: f for f in iter_features(countries)}

def draw_country(iso3, style=None):
    """Helper: draw one country centred on its bounding box."""
    feature = feature_by_iso3[iso3]
    name    = feature['properties']['ADMIN']
    bbox    = bbox_from_feature(feature)
    print(f'{name}  bbox: {tuple(round(x,1) for x in bbox)}')
    m = make_map()
    add_geojson(m, feature, style=style)
    fit_map_to_geojson(m, feature)
    return m

In [7]:
draw_country('CHL')

Chile  bbox: (-109.5, -55.9, -66.4, -17.5)


Map(center=[20, 10], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

In [8]:
draw_country('RUS')

Russia  bbox: (-180.0, 41.2, 180.0, 81.9)


Map(center=[20, 10], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

In [9]:
draw_country('NRU')

Nauru  bbox: (166.9, -0.6, 167.0, -0.5)


Map(center=[20, 10], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

---
## Microlesson 4 — Find the middle of a country

Both methods are implemented in `wdo.games.worldle.feature_center`.
The sanity-check below plots a red dot at every country's centre;
scan for dots in the ocean.


In [10]:
from wdo.games.worldle import feature_center

for iso3, label in [('RUS','Russia'), ('USA','United States of America'),
                     ('CHL','Chile'), ('NRU','Nauru')]:
    if iso3 not in feature_by_iso3:
        continue
    f = feature_by_iso3[iso3]
    bbox_c = feature_center(f, method='bbox')
    mean_c = feature_center(f, method='mean')
    print(f'{label:30s}  bbox={bbox_c}   mean={mean_c}')

Russia                          bbox=(61.52569529500013, 9.947598300641403e-14)   mean=(63.9091053068347, 86.96823029676617)
United States of America        bbox=(45.159309744500106, 0.3187158540001178)   mean=(48.05276503907437, -120.48611620338407)
Chile                           bbox=(-36.71254621049989, -87.93726559149991)   mean=(-47.20624659489612, -72.85561332907803)
Nauru                           bbox=(-0.5211320944999115, 166.9326278005001)   mean=(-0.5177819424443615, 166.93537664288894)


In [11]:
from ipyleaflet import CircleMarker

valid_features = [f for f in iter_features(countries)
                  if f['properties']['ISO_A3'] != '-99']

centre_map = make_map(center=(20, 10), zoom=2)
for feat in valid_features:
    lat, lon = feature_center(feat, method='bbox')
    centre_map.add(CircleMarker(
        location=(lat, lon), radius=3,
        color='red', fill_color='red', fill_opacity=0.7, weight=1,
    ))
centre_map

Map(center=[20, 10], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

---
## Microlesson 5 — Pick a target country


In [12]:
from wdo.games.worldle import choose_target

target = choose_target(valid_features, seed=42)
print('seed=42 target:', target['properties']['ADMIN'])

target_random = choose_target(valid_features)
print('random target:', target_random['properties']['ADMIN'])

seed=42 target: Pakistan
random target: New Caledonia


---
## Microlesson 6 — Judge the guess

**Double-check direction:** Brazil → France should give a NE arrow.


In [13]:
from wdo.games.worldle import format_feedback, guess_feedback

brazil = feature_by_iso3['BRA']
france = feature_by_iso3['FRA']

fb = guess_feedback(brazil, france)
print('Feedback dict:', fb)
print('Formatted   :', format_feedback(fb))
print('In miles    :', format_feedback(fb, units='miles'))


Feedback dict: {'correct': False, 'distance_km': 6236.0, 'distance_miles': 3874.9, 'bearing_deg': 60.7, 'compass': 'NE', 'arrow': '↗'}
Formatted   : 6,236 km away  ↗ NE
In miles    : 3,875 mi away  ↗ NE


---
## Microlesson 7 — Render one guess row

The flag is embedded as a base64 data-URI (portable — works even if the
notebook moves). Falls back to `flagcdn.com` if the local SVG is missing.


In [14]:
from IPython.display import HTML, display
from wdo.games.worldle import render_guess_row

meta = country_lookup['BRA']
row_html = render_guess_row(
    country_name = 'Brazil',
    flag_path    = meta['flag_path'],
    arrow        = fb['arrow'],
    distance_km  = fb['distance_km'],
    flag_base_dir= FLAG_DIR,
    guess_number = 1,
)
display(HTML(row_html))


---
## Microlesson 8 — Full game UI

Everything wired together: map → dropdown → Guess button → growing history.
State lives in `WorldleGame`, not scattered globals.


In [15]:
from ipyleaflet import GeoJSON
from ipywidgets import Button, Combobox, HTML, HBox, Layout, Output, VBox

_GREEN_SQ  = "\U0001f7e9"
_WHITE_SQ  = "⬜"
_YELLOW_SQ = "\U0001f7e8"
_RED_SQ    = "\U0001f7e5"
_CHECK     = "✅"
_CROSS     = "❌"
_PARTY     = "\U0001f389"
_SAD       = "\U0001f614"
_GLOBE     = "\U0001f30d"
_DASH      = "—"

_TIER_LABEL = {"easy": "🟢 Easy", "medium": "🟡 Medium", "hard": "🔴 Hard", None: "✨ Any"}

name_to_iso3         = {f['properties']['ADMIN']: f['properties']['ISO_A3']
                        for f in valid_features}
country_names_sorted = sorted(name_to_iso3)


class WorldleGame:
    """All state and UI for one round of Worldle.

    Pass seed= for a reproducible round — share the number with a classmate
    so they can play the same country.

    Pass difficulty= ('easy', 'medium', 'hard', or None) to restrict the
    random pool to countries in that area tier.  None (default) draws from
    all playable countries.
    """

    MAX_GUESSES = 6

    def __init__(self, seed=None, difficulty=None):
        self.seed        = seed
        self.difficulty  = difficulty
        self.target      = choose_target(valid_features, seed=seed, difficulty=difficulty)
        self.target_iso3 = self.target['properties']['ISO_A3']
        self.target_name = self.target['properties']['ADMIN']
        self.guesses     = []
        self.finished    = False
        self._build_ui()

    def _build_ui(self):
        self.map = make_map()
        add_geojson(self.map, self.target)
        fit_map_to_geojson(self.map, self.target)

        self.combobox = Combobox(
            options=country_names_sorted,
            placeholder='Type a country name…',
            ensure_option=True,
            layout=Layout(width='260px'),
        )
        self.guess_btn    = Button(description='Guess',    button_style='primary', layout=Layout(width='80px'))
        self.give_up_btn  = Button(description='Give up',  button_style='warning', layout=Layout(width='80px'))
        self.new_game_btn = Button(description='New game', button_style='info',    layout=Layout(width='90px'))
        self.progress     = HTML(value=self._progress_html())
        self.banner       = Output()
        self.history      = Output()

        self.guess_btn.on_click(self._on_guess)
        self.give_up_btn.on_click(self._on_give_up)
        self.new_game_btn.on_click(self._on_new_game)
        self.combobox.observe(lambda ch: self.banner.clear_output(wait=True), names='value')

    def _progress_html(self):
        n   = len(self.guesses)
        rem = self.MAX_GUESSES - n
        dots = _GREEN_SQ * n + _WHITE_SQ * rem
        diff_badge = ""
        if self.difficulty:
            diff_badge = (f'<span style="margin-left:10px;font-size:11px;'
                          f'background:#f0f0f0;padding:2px 8px;border-radius:8px;'
                          f'color:#555">{_TIER_LABEL[self.difficulty]}</span>')
        return f'<span style="font-size:15px;line-height:32px">{dots}</span>{diff_badge}'

    def _on_guess(self, _):
        if self.finished:
            return
        name = (self.combobox.value or '').strip()
        if not name or name not in name_to_iso3:
            with self.banner:
                self.banner.clear_output(wait=True)
                display(HTML('<p style="color:#e76f51;font-family:system-ui;margin:4px 0">'
                             'Pick a country from the list.</p>'))
            return

        iso3 = name_to_iso3[name]
        if iso3 in {g['iso3'] for g in self.guesses}:
            with self.banner:
                self.banner.clear_output(wait=True)
                display(HTML('<p style="color:#e76f51;font-family:system-ui;margin:4px 0">'
                             'Already guessed that one!</p>'))
            return

        fb   = guess_feedback(feature_by_iso3[iso3], self.target)
        meta = country_lookup.get(iso3, {})
        self.guesses.append({'iso3': iso3, 'name': name, 'feedback': fb})

        with self.history:
            display(HTML(render_guess_row(
                country_name  = name,
                flag_path     = meta.get('flag_path'),
                arrow         = fb['arrow'],
                distance_km   = fb['distance_km'],
                flag_base_dir = FLAG_DIR,
                guess_number  = len(self.guesses),
            )))

        self.progress.value = self._progress_html()
        self.combobox.value = ''
        self.banner.clear_output()

        if fb['correct']:
            self._end_game(won=True)
        elif len(self.guesses) >= self.MAX_GUESSES:
            self._end_game(won=False)

    def _on_give_up(self, _):
        if not self.finished:
            self._end_game(won=False)

    def _on_new_game(self, _):
        new = WorldleGame(seed=None, difficulty=self.difficulty)
        with _container:
            _container.clear_output(wait=True)
            new._render()

    def _end_game(self, won):
        self.finished             = True
        self.combobox.disabled    = True
        self.guess_btn.disabled   = True
        self.give_up_btn.disabled = True

        self.map.add(GeoJSON(data=self.target, style={
            'color': '#e63946', 'fillColor': '#e63946',
            'weight': 3, 'fillOpacity': 0.55,
        }))
        fit_map_to_geojson(self.map, self.target)

        meta = country_lookup.get(self.target_iso3, {})
        from wdo.games.worldle import _flag_src as get_src
        fsrc = get_src(meta.get('flag_path'), FLAG_DIR)
        flag_html = (
            f'<img src="{fsrc}" height="26" style="vertical-align:middle;'
            'margin:0 5px;border:1px solid #ddd;border-radius:2px">'
            if fsrc else ''
        )
        n = len(self.guesses)

        if won:
            msg = (f'<div style="font-family:system-ui;padding:8px 0;font-size:16px">'
                   f'{_PARTY} <strong style="color:#2a9d8f">You got it!</strong> '
                   f'{flag_html}<strong>{self.target_name}</strong> '
                   f'{_DASH} {n} guess{"" if n == 1 else "es"}</div>')
        else:
            msg = (f'<div style="font-family:system-ui;padding:8px 0;font-size:16px">'
                   f'{_SAD} The answer was {flag_html}'
                   f'<strong>{self.target_name}</strong>.</div>')

        with self.banner:
            self.banner.clear_output(wait=True)
            display(HTML(msg))

        print(self._share_text(won))

    def _share_text(self, won):
        """Emoji-grid result string, copy-paste friendly."""
        mark  = _CHECK if won else _CROSS
        diff_str = _TIER_LABEL.get(self.difficulty, "✨ Any")
        lines = [f"Worldle {mark} {len(self.guesses)}/{self.MAX_GUESSES}  {diff_str}"]
        for g in self.guesses:
            fb  = g['feedback']
            if fb['correct']:
                box = _GREEN_SQ
            elif fb['distance_km'] < 2000:
                box = _YELLOW_SQ
            else:
                box = _RED_SQ
            lines.append(f"{box} {g['name']} {fb['arrow']} {fb['distance_km']:,.0f} km")
        return '\n'.join(lines)

    def _render(self):
        """Build and display the full widget tree (must be called inside an Output context)."""
        diff_str = _TIER_LABEL.get(self.difficulty, "✨ Any")
        header = HTML(
            f'<div style="font-family:-apple-system,system-ui,sans-serif;'
            f'border-bottom:2px solid #2a9d8f;padding-bottom:6px;margin-bottom:8px">'
            f'<span style="font-size:24px;font-weight:700">{_GLOBE} Worldle</span>'
            f'<span style="font-size:13px;color:#888;margin-left:12px">'
            f'Guess the mystery country {_DASH} 6 tries</span>'
            f'<span style="font-size:12px;background:#f0f0f0;padding:2px 8px;'
            f'border-radius:8px;margin-left:10px;color:#555">{diff_str}</span></div>'
        )
        col_header = HTML(
            '<div style="display:flex;gap:10px;padding:3px 10px;'
            'font-size:12px;color:#bbb;font-family:system-ui;border-bottom:1px solid #eee">'
            '<span style="width:22px">#</span><span style="width:36px">Flag</span>'
            '<span style="flex:1">Country</span><span>Dir</span>'
            '<span style="min-width:95px;text-align:center">Distance</span>'
            '<span style="width:20px"></span></div>'
        )
        controls = HBox(
            [self.combobox, self.guess_btn, self.give_up_btn,
             self.new_game_btn, self.progress],
            layout=Layout(align_items='center', margin='6px 0'),
        )
        display(VBox([header, self.map, controls,
                      self.banner, col_header, self.history]))

    def show(self):
        """Entry point: render into the shared container and display it."""
        with _container:
            _container.clear_output(wait=True)
            self._render()
        display(_container)

---
## Microlesson 9 — Polish

Polish features included in this game:

1. **Proximity colour-coding** — distance badge changes colour: teal < 500 km, amber < 2 000 km, coral ≥ 2 000 km.
2. **Proximity emoji** — 🔥 / 🌡️ / 🧊 per row, same thresholds.
3. **6-guess limit** with 🟩/⬜ progress tracker (Wordle-style).
4. **Country reveal on the map** at game end (red overlay).
5. **Shareable result string** printed to console (emoji grid, copy-paste friendly).
6. **Seed-based reproducibility** — share the seed to challenge a classmate.
7. **New Game button** replaces UI in-place without adding a new cell output.


In [16]:
from wdo.geometry.area import difficulty_tier, polygon_area_km2, tier_label, EASY_THRESHOLD, MEDIUM_THRESHOLD

area_data = []
for feat in valid_features:
    area   = polygon_area_km2(feat)
    tier   = difficulty_tier(area)
    name   = feat['properties']['ADMIN']
    iso3   = feat['properties']['ISO_A3']
    area_data.append({'name': name, 'iso3': iso3, 'area_km2': area, 'tier': tier})

area_data.sort(key=lambda d: d['area_km2'], reverse=True)

print("Top 5 largest:")
for d in area_data[:5]:
    print(f"  {d['name']:<35} {d['area_km2']:>14,.0f} km²  [{tier_label(d['tier'])}]")

print("\nBottom 5 smallest:")
for d in area_data[-5:]:
    print(f"  {d['name']:<35} {d['area_km2']:>14,.0f} km²  [{tier_label(d['tier'])}]")

tiers  = {'easy': [], 'medium': [], 'hard': []}
for d in area_data:
    tiers[d['tier']].append(d['name'])

print(f"\nTier counts (out of {len(area_data)} playable features):")
for t in ('easy', 'medium', 'hard'):
    print(f"  {tier_label(t):<18}  {len(tiers[t]):>3} countries  "
          f"e.g. {', '.join(tiers[t][:4])}")

print(f"\nThresholds: easy >= {EASY_THRESHOLD:,} km²  |  medium >= {MEDIUM_THRESHOLD:,} km²")
print(f"Ratio  easy : medium : hard  =  1 : {len(tiers['medium'])/len(tiers['easy']):.1f} : {len(tiers['hard'])/len(tiers['easy']):.1f}")

Top 5 largest:
  Russia                                  16,859,271 km²  [🟢 Easy]
  Antarctica                              12,258,370 km²  [🟢 Easy]
  Canada                                   9,894,398 km²  [🟢 Easy]
  United States of America                 9,447,941 km²  [🟢 Easy]
  China                                    9,372,761 km²  [🟢 Easy]

Bottom 5 smallest:
  Sint Maarten                                    23 km²  [🔴 Hard]
  Tuvalu                                          23 km²  [🔴 Hard]
  Monaco                                          19 km²  [🔴 Hard]
  Gibraltar                                        4 km²  [🔴 Hard]
  Vatican                                          0 km²  [🔴 Hard]

Tier counts (out of 238 playable features):
  🟢 Easy               31 countries  e.g. Russia, Antarctica, Canada, United States of America
  🟡 Medium             76 countries  e.g. United Republic of Tanzania, Venezuela, Nigeria, Pakistan
  🔴 Hard              131 countries  e.g. South Korea, 

In [17]:
from ipywidgets import ToggleButtons

_DIFF_OPTIONS = ['✨ Any', '🟢 Easy', '🟡 Medium', '🔴 Hard']
_DIFF_MAP     = {'✨ Any': None, '🟢 Easy': 'easy', '🟡 Medium': 'medium', '🔴 Hard': 'hard'}

diff_selector = ToggleButtons(
    options=_DIFF_OPTIONS,
    value='✨ Any',
    description='Difficulty:',
    button_style='',
    style={'description_width': 'initial', 'button_width': '90px'},
)

start_btn = Button(
    description='▶  Start',
    button_style='success',
    layout=Layout(width='110px', height='36px', margin='6px 0 0 8px'),
)

_container = Output()

def _on_start(_):
    chosen_difficulty = _DIFF_MAP[diff_selector.value]
    with _container:
        _container.clear_output(wait=True)
        game = WorldleGame(seed=None, difficulty=chosen_difficulty)
        game._render()

start_btn.on_click(_on_start)

display(VBox([
    HTML('<div style="font-family:system-ui;font-size:13px;color:#555;margin-bottom:4px">'
         'Pick a difficulty then click Start (or re-run the cell for a new game).</div>'),
    HBox([diff_selector, start_btn]),
    _container,
]))